<a href="https://colab.research.google.com/github/annaBANNANA6/protein-interaction-network/blob/main/HINT_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Importing data**

*   `Uniprot_A` and `Uniprot_B`: standard identifiers for each protein in the pair
*   `Gene_A` and `Gene_B`
*   `pmid:method:quality:type`: each entry encodes PubMedID, experimental method, quality flag, interaction type together (seperated by colons)

--> if backed by both LC and HT, include both, separated by  '|' divider


*   `taxid`: NCBI taxonomy ID (confirms which orgnaism the interaction belongs to, human = 9606)
* `high_quality`: boolean confirming _hq values
* `in_pdb_source / PDB_IDs`: whether interaction has 3D structural evidence in Protein Data Bank, which structure ID if so





In [2]:
import pandas as pd
data = pd.read_csv("https://hint.yulab.org/download-raw/2024-06/HomoSapiens_binary_hq.txt", sep = "\t")

print(f"shape: {data.shape}") # should be 163435, 9 columns
print(f"\ncolumns: {data.columns}")

data.head()

shape: (163435, 9)

columns: Index(['Uniprot_A', 'Uniprot_B', 'Gene_A', 'Gene_B',
       'pmid:method:quality:type', 'taxid', 'high_quality', 'in_pdb_source',
       'PDB_IDs'],
      dtype='object')


,Uniprot_A,Uniprot_B,Gene_A,Gene_B,pmid:method:quality:type,taxid,high_quality,in_pdb_source,PDB_IDs
0,A0A024R2I8,F1D8Q5,NR1A2,NR2B1,15604093:0018:HT:binary|19211732:0018:LC:binar...,9606,True,False,NaN
1,A0A024R2I8,F1D8Q7,NR1A2,NR2B3,15604093:0018:HT:binary,9606,True,False,NaN
2,A0A024R2I8,O75376,NR1A2,NCOR1,12799135:0018:LC:binary|14985366:0096:LC:binar...,9606,True,False,NaN
3,A0A024R2I8,P12931,NR1A2,SRC,10454579:0096:LC:binary|14985366:0096:LC:binar...,9606,True,False,NaN
4,A0A024R2I8,P28702,NR1A2,RXRB,15604093:0018:HT:binary|9346901:0096:LC:binary,9606,True,False,NaN


**data orientation**

Is `taxid` uniform (all human)?
Is `high_quality` all True?
How many rows have null for PDB_Ids (which interactions have structural evidence)?

In [20]:
print(data['taxid'].unique()) # confirmed all human

print(data["high_quality"].unique()) # confirmed --> note to strip the space preceding this value

# data[data["PDB_IDs"].isna()]
print(f"# null rows for PDB_IDs: {(data["PDB_IDs"].isna()).sum()}")

[9606]
[ True]
# null rows for PDB_IDs: 149855


so the results: all human, all high quality, and 149855/163435 of rows are null for PDB_IDs, meaning that most are missing 3D strutural info

---

**data cleaning**
note that due to nature of HINT's purpose to remove erroneous and low-quality entries, there is not much tabular data cleaning checks for this (and already checked taxid and quality).

network specific cleaning checks:
*   **Self-loops** (does any rows have Uniprot_A == Uniprot_B? this would biologically that a protein interacting with itself, which does happen but might not want to keep depending on research question
*   **duplicate or reciprocal edges** are there rows where A,B appears once and B,A appears as separate row? check since PPIs lack direction, so these are the same



In [26]:
# self loop checks
self_loops = data[data["Uniprot_A"] == data["Uniprot_B"]]
print(f"Number of self loops / rows where A = B: {self_loops.shape[0]}")

# duplicate/reciprocal edges
# print(f"duplicates: {data.duplicated().sum()}")

# smaller in protein_1, larger in protein_2 (alphabetical comparison)
protein_1 = []
protein_2 = []

for index, row in data.iterrows(): # iterrow() - loop over df by row, each row is dict you index by column name
  a = row["Uniprot_A"]
  b = row["Uniprot_B"]

  if a < b:
    protein_1.append(a)
    protein_2.append(b)
  else:
    protein_1.append(b)
    protein_2.append(a)

# print(protein_1)
# print(protein_2)

# add sorted AB/BA columns back to DF and check duplicates
data["protein_1"] = protein_1
data["protein_2"] = protein_2
print(f"duplicates: {data.duplicated(subset = ["protein_1", "protein_2"]).sum()}")



Number of self loops / rows where A = B: 6034
duplicates: 0


In [28]:
# alternative way to check duplicates using np methods
import numpy as np
data["protein_1"] = np.minimum(data["Uniprot_A"], data["Uniprot_B"])
data["protein_2"] = np.maximum(data["Uniprot_A"], data["Uniprot_B"])
print(f"duplicates: {data.duplicated(subset = ["protein_1", "protein_2"]).sum()}")

duplicates: 0


**results**: 0 duplicate rows, but 6034 self loops, so protein interacting with itself (**homodimeritzation** which is mechanism where some transcription factors and enzymes ONLY funciton when two copies of themselves bind together. even with bio significance, think about mathematical significance --> drop to avoid ambiguity of adding degree to protein,inflating clustering ocefficient (loops create triangles AAB)

---

In [18]:
# dropping self loops for topological analysis
data.drop(self_loops.index)

,Uniprot_A,Uniprot_B,Gene_A,Gene_B,pmid:method:quality:type,taxid,high_quality,in_pdb_source,PDB_IDs
0,A0A024R2I8,F1D8Q5,NR1A2,NR2B1,15604093:0018:HT:binary|19211732:0018:LC:binar...,9606,True,False,NaN
1,A0A024R2I8,F1D8Q7,NR1A2,NR2B3,15604093:0018:HT:binary,9606,True,False,NaN
2,A0A024R2I8,O75376,NR1A2,NCOR1,12799135:0018:LC:binary|14985366:0096:LC:binar...,9606,True,False,NaN
3,A0A024R2I8,P12931,NR1A2,SRC,10454579:0096:LC:binary|14985366:0096:LC:binar...,9606,True,False,NaN
4,A0A024R2I8,P28702,NR1A2,RXRB,15604093:0018:HT:binary|9346901:0096:LC:binary,9606,True,False,NaN
...,...,...,...,...,...,...,...,...,...
163361,Q9Y5Z9,Q9Y680,UBIAD1,FKBP7,32296183:0018:HT:binary|32296183:0397:HT:binar...,9606,True,False,NaN
163383,Q9Y6D5,Q9Y6D6,ARFGEF2,ARFGEF1,10716990:0006:LC:binary|10716990:0071:LC:binar...,9606,True,False,NaN
163386,Q9Y6D9,Q9Y6W3,MAD1L1,CAPN7,32296183:0018:HT:binary|32296183:0397:HT:binar...,9606,True,False,NaN
163419,S6B2B6,S6C4R2,NaN,NaN,29760382:0114:LC:binary,9606,True,True,5W1K-O:T


**build network graph**: once edges become graph object, check for **isolated nodes** and **connected components**
*   **isolated node:** degree 0. consider that graph is being built from edge list itself so each node has at least on edge
*  **connected components**: component (cluster of nodes accessible from each other). usually get ONE big component with most proteins (small world effect) + long tail of smaller disconnected fragments (small clusters only interact w/ e/o

relevance: betwness centrality, shortest path length only WIHTIN compoment, so start analysis on giant component and note to do separately on smaller ones



In [ ]:
# "networkx from pandas edgelist"" to create Graph
# network function related to connected components to extract them and to check if wholeg graph connected piece
# size of each component (combine components function with len() in loop)
# isolated nodes: "networks degree 0", G.degree()


Self-loops — a node connected to itself (A-A). Usually removed unless self-interaction is biologically meaningful to your question (e.g., homodimerization).
2. Duplicate/parallel edges — the same edge appearing more than once. For undirected graphs like PPIs, this includes reciprocal duplicates (A-B and B-A as separate rows representing one edge).
3. Isolated nodes — proteins that appear in your node list (e.g., if you're merging with an external gene list) but have zero edges. These usually get dropped for network-metric calculations since degree/centrality are meaningless at 0, but worth checking whether any snuck in.
4. Disconnected components — most real biological networks have one giant connected component and then a bunch of small, disconconnected fragments (pairs or small clusters floating off on their own). Common practice is to check how many components exist and how large the largest one is — many analyses (like betweenness centrality) only make sense on the giant component, since centrality is undefined between disconnected nodes.
5. Node ID consistency — make sure every node is identified consistently (you have both UniProt IDs and gene names — pick one as your canonical node ID and stick with it, since mixing them creates phantom duplicate nodes).
6. Multi-edge attribute conflicts — when the same edge appears from multiple studies with different quality/type/method values (like your pmid:method:quality:type field), you'll need a rule for how to collapse that into one edge — do you keep all evidence as edge metadata, or do you just care that an edge exists at all?
Checks 1–3 are close to universal — almost every network project does these. Checks 4–6 become more important specifically because you're merging identifiers and possibly external data (like the node classification path would require).